In [2]:

import pandas as pd
import glob
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. Encontrar o dataset da Olist
arquivos = glob.glob(
    "/kaggle/input/**/olist_order_reviews_dataset.csv",
    recursive=True
)

if not arquivos:
    raise FileNotFoundError(
        "Arquivo de avaliações não encontrado no Input."
    )

df = pd.read_csv(arquivos[0])

# 2. Selecionar comentários e notas
df = df[["review_comment_message", "review_score"]]
df = df.dropna()

# 3. Definir sentimentos
# Notas 1 e 2 = negativo
# Nota 3 = neutro (não utilizada)
# Notas 4 e 5 = positivo
df = df[df["review_score"] != 3].copy()
df["sentimento"] = (df["review_score"] >= 4).astype(int)

# 4. Separar dados de treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    df["review_comment_message"],
    df["sentimento"],
    test_size=0.2,
    random_state=42,
    stratify=df["sentimento"]
)

# 5. Criar e treinar o modelo
modelo = make_pipeline(
    TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2)
    ),
    LogisticRegression(max_iter=1000)
)

modelo.fit(X_train, y_train)

# 6. Avaliar o desempenho
previsoes = modelo.predict(X_test)

print("Acurácia:", round(accuracy_score(y_test, previsoes), 4))
print(classification_report(y_test, previsoes))

# 7. Testar uma avaliação
comentario = ["A comida chegou rápida e estava deliciosa"]

resultado = modelo.predict(comentario)[0]

print("Comentário:", comentario[0])
print("Sentimento:", "Positivo" if resultado == 1 else "Negativo")

# 8. Salvar o modelo
joblib.dump(
    modelo,
    "/kaggle/working/modelo_sentimentos_olist.joblib"
)

print("Modelo salvo com sucesso!")


Acurácia: 0.929
              precision    recall  f1-score   support

           0       0.88      0.87      0.88      2178
           1       0.95      0.95      0.95      5306

    accuracy                           0.93      7484
   macro avg       0.92      0.91      0.91      7484
weighted avg       0.93      0.93      0.93      7484

Comentário: A comida chegou rápida e estava deliciosa
Sentimento: Positivo
Modelo salvo com sucesso!
